## Important: Makesure torchscale import is the custom version (so path should be relative to this project's folder), not the pip installed version.

In [1]:
import torchscale
print(torchscale.__file__)

C:\Users\Za\Desktop\SchoolWork\GMU\757 GAN\retnet_hf_wrapper\scripts\torchscale\__init__.py


In [2]:
#Get CPU Count to speed up the tokenize part
import multiprocessing
num_cpu_proc = max (6, multiprocessing.cpu_count())

#Configure Batch_Size and learning rate based on GPU Vram.
from types import SimpleNamespace
args = SimpleNamespace() #Initialize
import torch
available_gpu_memory = torch.cuda.mem_get_info()[0] / (1024**3) #Get available gpu memory in Byte, convert to GB.
print ("available_gpu_memory: ", available_gpu_memory)
args.__dict__.update({
    "per_device_train_batch_size":round(available_gpu_memory),
    "per_device_eval_batch_size":round(available_gpu_memory),
    "gradient_accumulation_steps":2,
    "bf16": False,
    "fp16": True,
    #epoch/lr/weight decay
    "num_train_epochs": 1 * round(available_gpu_memory**0.4), #Feng Shui told me
    "learning_rate": 1e-4 * round(available_gpu_memory**0.5), #Feng Shui told me
    "weight_decay": 0.01,
})

args.__dict__.update({
    #Logging Location
    "dataset_name": "FiscalNote/billsum",
    "output_dir": "/content/drive/MyDrive/CS757/model_checkpoint/retnet-billsum",

    #Logging/Saving/Evaluating
    "logging_steps": 20,
    "save_steps": 2500,
    "eval_steps": 200,
    "dataset_config_name": None,
    "dataset_text_field": None,
    "streaming": False,

    #Basic training config
    "max_steps": -1,
    "warmup_ratio": 0.03,
    "tokenizer_name": "gpt2",
    "dropout": 0.1,
    "activation_dropout": 0.0,

    #For trainer use only
    "block_size": 2048, #Chunking
    "max_train_samples": None,
    "max_eval_samples": 1024,
    "validation_split_percentage": 5,
    "seed": 42,
    "push_to_hub": False,
    "hub_model_id": None,

    #Model config
    
    # 2080Ti
    #"hidden_size": 512,
    
    #A100
    "hidden_size": 1024,
    
    "intermediate_size": 2048,
    "num_hidden_layers": 8,
    "num_retention_heads": 8,
    
    "value_dim": None,
    "drop_path_rate": 0.0,
    "recurrent_chunk_size": 128,
    "chunkwise_recurrent": True,  # IMPORTANT, TURNING ON CHUNKWISE MODE, Do not pass Cache into model.
})

available_gpu_memory:  9.523902870714664


In [3]:
from retnet_hf import RetNetConfig, RetNetForCausalLM

import importlib
import sys
import retnet_hf
importlib.reload(sys.modules['retnet_hf'])
importlib.reload(retnet_hf)
sys.modules["retnet_hf"] = retnet_hf


In [4]:
prompt = """SECTION 1. SHORT TITLE.

    This Act may be cited as the ``Government Shutdown Prevention
Act''.

SEC. 2. CONTINUING FUNDING.

    (a) In General.--If any regular appropriation bill for any fiscal
year does not become law prior to the beginning of these fiscal years
or a joint resolution making continuing appropriations is not in
effect, there is appropriated, out of any moneys in the Treasury not
otherwise appropriated, and out of applicable corporate or other
revenues, receipts, and funds, such sums as may be necessary to
continue any program, project, or activity for which funds were
provided in those fiscal years.
    (b) Level of Funding.--Appropriations and funds made available, and
authority granted, for a program, project, or activity for any fiscal
year pursuant to this Act shall be at 100 percent of the rate of
operations that was provided for the program, project, or activity in
fiscal year 1999 in the corresponding regular appropriation Act for
fiscal year 1999.
    (c) Period of Availability.--Appropriations and funds made
available, and authority granted, for any fiscal year pursuant to this
Act for a program, project, or activity shall be available for the
period beginning with the first day of a lapse in appropriations and
ending with the earlier of--
            (1) the date on which the applicable regular appropriation
        bill for any fiscal year becomes law (whether or not that law
        provides for that program, project, or activity) or a
        continuing resolution making appropriations becomes law, as the
        case may be; or
            (2) the last day of any fiscal year.

SEC. 3. TERMS AND CONDITIONS.

    (a) In General.--An appropriation of funds made available, or
authority granted, for a program, project, or activity for any fiscal
year pursuant to this Act shall be made available to the extent and in
the manner which would be provided by the pertinent appropriations Act
for fiscal year 1999, including all of the terms and conditions and the
apportionment schedule imposed with respect to the appropriation made
or funds made available for fiscal year 1999 or authority granted for
the program, project, or activity under current law.
    (b) Extent and Manner.--Appropriations made by this Act shall be
available to the extent and in the manner which would be provided by
the pertinent appropriations Act.

SEC. 4. COVERAGE.

    Appropriations and funds made available, and authority granted, for
any program, project, or activity for any fiscal year pursuant to this
Act shall cover all obligations or expenditures incurred for that
program, project, or activity during the portion of any fiscal year for
which this Act applies to that program, project, or activity.

SEC. 5. EXPENDITURES.

    Expenditures made for a program, project, or activity for any
fiscal year pursuant to this Act shall be charged to the applicable
appropriation, fund, or authorization whenever a regular appropriation
bill or a joint resolution making continuing appropriations until the
end of any fiscal year providing for that program, project, or activity
for that period becomes law.

SEC. 6. INITIATING OR RESUMING A PROGRAM, PROJECT, OR ACTIVITY.

    No appropriation or funds made available or authority granted
pursuant to this Act shall be used to initiate or resume any program,
project, or activity for which appropriations, funds, or other
authority were not available during fiscal year 1999.

SEC. 7. PROTECTION OF OTHER OBLIGATIONS.

    Nothing in this Act shall be construed to effect Government
obligations mandated by other law, including obligations with respect
to Social Security, Medicare, Medicaid, and veterans benefits.

SEC. 8. DEFINITION.

    In this Act, the term ``regular appropriation bill'' means any
annual appropriation bill making appropriations, otherwise making funds
available, or granting authority, for any of the following categories
of programs, projects, and activities:
            (1) Agriculture, rural development, and related agencies
        programs.
            (2) The Departments of Commerce, Justice, and State, the
        judiciary, and related agencies.
            (3) The Department of Defense.
            (4) The government of the District of Columbia and other
        activities chargeable in whole or in part against the revenues
        of the District.
            (5) The Departments of Labor, Health and Human Services,
        and Education, and related agencies.
            (6) The Departments of Veterans Affairs and Housing and
        Urban Development, and sundry independent agencies, boards,
        commissions, corporations, and offices.
            (7) Energy and water development.
            (8) Foreign assistance and related programs.
            (9) The Department of the Interior and related agencies.
            (10) Military construction.
            (11) The Department of Transportation and related agencies.
            (12) The Treasury Department, the U.S. Postal Service, the
        Executive Office of the President, and certain independent
        agencies.
            (13) The legislative branch.
Summary:"""

In [5]:
from transformers import AutoTokenizer, StaticCache, DynamicCache
from retnet_hf import RetNetConfig, RetNetForCausalLM
import torch, os
import safetensors.torch

#### Model/Tokenizer Loading

# Path to checkpoint
#checkpoint_path = "./outputs/retnet-billsum/checkpoint-42835"
checkpoint_path = "./outputs/retnet-billsum-A100 Checkpoint/checkpoint-21420"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

config = RetNetConfig(
    vocab_size=len(tokenizer),
    hidden_size=args.hidden_size,
    intermediate_size=args.intermediate_size,
    num_hidden_layers=args.num_hidden_layers,
    num_retention_heads=args.num_retention_heads,
    value_dim=args.value_dim,
    hidden_dropout_prob=args.dropout,
    activation_dropout=args.activation_dropout,
    drop_path_rate=args.drop_path_rate,
    recurrent_chunk_size=args.recurrent_chunk_size,
    chunkwise_recurrent=args.chunkwise_recurrent,
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

model = RetNetForCausalLM(config)
model.resize_token_embeddings(len(tokenizer))
state_dict = safetensors.torch.load_file(os.path.join(checkpoint_path, "model.safetensors"), device="cpu")
load_result = model.load_state_dict(state_dict, strict=False)

# Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

[transformers] You are using a model of type `retnet` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


RetNetForCausalLM(
  (model): RetNetModel(
    (embed_tokens): Embedding(50257, 1024)
    (decoder): RetNetDecoder(
      (dropout_module): Dropout(p=0.1, inplace=False)
      (embed_tokens): Embedding(50257, 1024)
      (output_projection): Linear(in_features=1024, out_features=50257, bias=False)
      (layers): ModuleList(
        (0-7): 8 x DecoderLayer(
          (dropout_module): Dropout(p=0.1, inplace=False)
          (retention): MultiScaleRetention(
            (q_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (g_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (group_norm): RMSNorm()
          )
          (retention_layer_norm): RMSNorm()
          (ffn): GLU(
            (activation_dropout_

In [6]:
# PARALLEL GENERATE
# Tokenize
inputs = tokenizer(prompt, return_tensors="pt").to(device)
# Generate
with torch.no_grad():
    """cache = StaticCache(
        config=model.config,
        batch_size=inputs["input_ids"].shape[0],
        max_cache_len=len(inputs["input_ids"][0]) * args.hidden_size//args.num_retention_heads,
        device=model.device,
        dtype=model.dtype,
    )"""
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        #temperature=1,
        #top_p=0.98,
        pad_token_id=tokenizer.eos_token_id,
        use_cache = False,
        #past_key_values=cache,
    )

# Decode
generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generated)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1376 > 1024). Running this sequence through the model will result in indexing errors


SECTION 1. SHORT TITLE.

    This Act may be cited as the ``Government Shutdown Prevention
Act''.

SEC. 2. CONTINUING FUNDING.

    (a) In General.--If any regular appropriation bill for any fiscal
year does not become law prior to the beginning of these fiscal years
or a joint resolution making continuing appropriations is not in
effect, there is appropriated, out of any moneys in the Treasury not
otherwise appropriated, and out of applicable corporate or other
revenues, receipts, and funds, such sums as may be necessary to
continue any program, project, or activity for which funds were
provided in those fiscal years.
    (b) Level of Funding.--Appropriations and funds made available, and
authority granted, for a program, project, or activity for any fiscal
year pursuant to this Act shall be at 100 percent of the rate of
operations that was provided for the program, project, or activity in
fiscal year 1999 in the corresponding regular appropriation Act for
fiscal year 1999.
    (c) Pe

In [8]:
# Tokenize
#os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
# Generate
with torch.no_grad():
    """cache = StaticCache(
        config=model.config,
        batch_size=inputs["input_ids"].shape[0],
        #max_cache_len=len(inputs["input_ids"][0]) * args.hidden_size//args.num_retention_heads, 
        max_cache_len=args.hidden_size//args.num_retention_heads, 
        device=model.device,
        dtype=model.dtype,
    )"""
    cache = None
    ## Warm up:
    print("warmup token: ")
    for i in range(len(inputs["input_ids"][0])):
        token_input = {
            'input_ids': inputs["input_ids"][:, i:i+1],
            'attention_mask': inputs["attention_mask"][:, i:i+1],
        }
        outputs = model(
            **token_input,
            #use_cache=True,
            past_key_values=cache,
            #cache_position=torch.tensor([i], device=device),
            return_dict=True,
        )
        next_token = outputs.logits[:, -1].argmax(dim=-1)
        print(tokenizer.decode(next_token[0]), end='')
    # Generation
    print("\ngenerating token: ")
    next_token = outputs.logits[:, -1].argmax(dim=-1)
    print(tokenizer.decode(next_token[0]), end='')
    inputs = {'input_ids': torch.tensor([[next_token]], device='cuda:0'), 'attention_mask': torch.tensor([[1]], device='cuda:0')}

    for i in range (0, 200):
        outputs = model(
            **inputs,
            temperature=1.25,
            top_p=0.98,
            use_cache=True,
            past_key_values=cache,
            return_dict=True,
        )
        next_token = outputs.logits[:, -1].argmax(dim=-1)
        #decoded = tokenizer.decode(next_token[0])
        print(tokenizer.decode(next_token[0]), end='')
        inputs = {'input_ids': torch.tensor([[next_token]], device='cuda:0'), 'attention_mask': torch.tensor([[1]], device='cuda:0')}

warmup token: 
. 1.
ORT TITLE.
     Act of be  as the or of Prevention Act  of
  .
.
U. FORD) FOR
    1) of general.--The the other basis, am the other year  period not be available, to the  on the  year after  the  resolution of a to to amended be the  on and are amended to and this the otherails, to the  to be  than  to and
 this the to conversion   ferredcy.-- and and and
. and  as the be  to the  to other to and. and . the the.   to the  year after
    1) of IV the.--Theliesations.--
. available to and
 ization Act the and the  to and. and . the other year  period to the section of be  the percent of the  of the ability.-- the made in the  to and. and . the ) year,, the  to basis, of the ) year,,
    1) ofic the.--Theliesations.--
. available  for and
 to the and the other year, to the section  of the  to and. and . be  to the   beginning on the   after the , the to
  the the  of the
            1) of  of the the  to basis,         am the other year, ,1 or  be the,         to the t

In [9]:
# Tokenize
#os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
# Generate
with torch.no_grad():
    #cache = DynamicCache()
    ## Warm up:
    print("warmup token: ")
    for i in range(len(inputs["input_ids"][0])):
        token_input = {
            'input_ids': inputs["input_ids"][:, i:i+1],
            'attention_mask': inputs["attention_mask"][:, i:i+1],
        }
        outputs = model(
            **token_input,
            #use_cache=True,
            #past_key_values=cache,
            cache_position=torch.tensor([i], device=device),
            return_dict=True,
        )
        next_token = outputs.logits[:, -1].argmax(dim=-1)
        print(tokenizer.decode(next_token[0]), end='')
    # Generation
    print("\ngenerating token: ")
    next_token = outputs.logits[:, -1].argmax(dim=-1)
    print(tokenizer.decode(next_token[0]), end='')
    inputs = {'input_ids': torch.tensor([[next_token]], device='cuda:0'), 'attention_mask': torch.tensor([[1]], device='cuda:0')}

    for i in range (0, 200):
        outputs = model(
            **inputs,
            temperature=1.25,
            top_p=0.98,
            #use_cache=True,
            #past_key_values=cache,
            return_dict=True,
        )
        next_token = outputs.logits[:, -1].argmax(dim=-1)
        #decoded = tokenizer.decode(next_token[0])
        print(tokenizer.decode(next_token[0]), end='')
        inputs = {'input_ids': torch.tensor([[next_token]], device='cuda:0'), 'attention_mask': torch.tensor([[1]], device='cuda:0')}

warmup token: 
. 1.
ORT TITLE.
     Act of be  as the or of Prevention Act  of
  .
.
U. FORD) FOR
    1) of general.--The the other basis, am the other year  period not be available, to the  on the  year after  the  resolution of a to to amended be the  on and are amended to and this the otherails, to the  to be  than  to and
 this the to conversion   ferredcy.-- and and and
. and  as the be  to the  to other to and. and . the the.   to the  year after
    1) of IV the.--Theliesations.--
. available to and
 ization Act the and the  to and. and . the other year  period to the section of be  the percent of the  of the ability.-- the made in the  to and. and . the ) year,, the  to basis, of the ) year,,
    1) ofic the.--Theliesations.--
. available  for and
 to the and the other year, to the section  of the  to and. and . be  to the   beginning on the   after the , the to
  the the  of the
            1) of  of the the  to basis,         am the other year, ,1 or  be the,         to the t

### Google Drive Specific Cells

In [ ]:
"""
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/CS757/retnet_hf" /content/ -r
#from retnet_hf import RetNetConfig, RetNetForCausalLM
#!mkdir output
!ls .
!pip install torchscale
!pip install fairscale
!cp "/content/drive/MyDrive/CS757/retnet_hf/multiscale_retention.py" "/usr/local/lib/python3.12/dist-packages/torchscale/component/multiscale_retention.py"
"""
"ay"